In [50]:
import vertexai 
import pandas as pd
from google import genai
import os
from search_eval_utils import (
    access_secret_version, 
    download_blob, 
    create_stratified_sample,
    process_and_save_prompt_output,
    prompt_v1, 
    jsonl_to_df
)

In [51]:
PROJECT_ID = 'proj-sales-recommender-dev'
LOCATION = 'us-central1'
BUCKET = 'sales_recommender_dev_bucket'
DATASET_NAME = "data/search_evaluation/true_search_labels.csv"
SEARCHES_DATASET_NAME = "data/search_evaluation/searches.csv"
SECRET_NAME = "gemini_api_key"
MODEL_ID = "gemini-2.0-flash-001"

In [53]:
# Init vertex 
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Create client using secret manager
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "key.json"
os.environ["GEMINI_API_KEY"] = access_secret_version(PROJECT_ID, SECRET_NAME)
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## Read and preprocess data from GCS

In [54]:
# Read data from GCS
os.makedirs("data", exist_ok=True)

download_blob(BUCKET, DATASET_NAME, 'data/true_search_labels.csv')
download_blob(BUCKET, SEARCHES_DATASET_NAME, 'data/searches.csv')
project_df = pd.read_csv('data/true_search_labels.csv')
searches_df = pd.read_csv('data/searches.csv')

In [55]:
# Drop unnecessary columns and LLMM generated columns
project_df = project_df.drop(columns=['Relevance', 'Reasoning', 'Query', 'FBM Relevance', 'FBM Notes', 'Search Name'])
project_df = project_df.drop_duplicates(subset=['ProjectID'], keep='first')

# Init search names
search_id_cols = ["Steel","FRP","Ceilings","Drywall","Core","EIFS","Insulation","Complimentary","Fry Reglet","Cultured Stone"]

# Split df into project data info and search names
search_ids = project_df[search_id_cols].drop(columns = ["Complimentary"])
project_df = project_df.drop(columns=search_id_cols)
search_ids.head()

,Steel,FRP,Ceilings,Drywall,Core,EIFS,Insulation,Fry Reglet,Cultured Stone
0,NaN,NaN,NaN,x,X,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,X,X,NaN,NaN,NaN,NaN
2,X,NaN,NaN,NaN,X,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,X,NaN,NaN


In [56]:
# Create a new DataFrame converting matrix into a single column with the search names
search_ids_df = pd.DataFrame()
search_ids_df['True Search Name'] = search_ids.apply(lambda row: ', '.join(col for col, val in row.items() if isinstance(val, str) and val.lower() == 'x'), axis=1)
search_ids_df['True Search Name'] = search_ids_df.applymap(lambda x: x.replace('Search ID', '').replace(":", "").strip())

# Add true search names the project_df
project_df['True Search Name'] = search_ids_df['True Search Name']
project_df = project_df.dropna(subset=['True Search Name'])
project_df = project_df[project_df['True Search Name'] != '']

/var/folders/t9/ynlywvys31d_71jprr25h4c00000gn/T/ipykernel_10065/1875629293.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  search_ids_df['True Search Name'] = search_ids_df.applymap(lambda x: x.replace('Search ID', '').replace(":", "").strip())


In [57]:
# Make a cross join between projects and searches
project_df_labeled = pd.merge(project_df, searches_df['SEARCH'], how = 'cross')

# Label YES/NO based on true search name and SEARCH  
project_df_labeled['True_Search_Label'] = project_df_labeled.apply(lambda x: 'YES' if x['SEARCH'] in x['True Search Name'] 
                                                                   else 'NO', axis=1)

In [58]:
# Save the labeled DataFrame to a CSV file
project_df_labeled = project_df_labeled[
    ['SEARCH', 'ProjectID', 'True_Search_Label', 
    'Title', 'Stage', 'Valuation_Value',
    'Valuation_Currency', 'Parameters_Parameter_Ownership',
    'Parameters_Parameter_WorkType', 'Parameters_Parameter_Structures',
    'DocumentAvailability_Plans', 'DocumentAvailability_Specs',
    'DocumentAvailability_Addenda', 'ParentCategories_PrimaryCategoryName',
    'ParentCategories_ParentCategory', 'Addresses_Address',
    'Details_Detail_Scope', 'Details_Detail_Notes', 'Details_Detail',
    'RSMeansMaterialDivisions_Division_Metals',
    'RSMeansMaterialDivisions_Division_ThermalandMoistureProtection',
    'RSMeansMaterialDivisions_Division_Openings',
    'RSMeansMaterialDivisions_Division_Finishes',
    'RSMeansMaterialDivisions_Division_Masonry', 'Details_Detail_Details',
    'Materials_Material', 'Notes_Note']]

project_df_labeled.to_csv('data/project_df_labeled.csv', index=False)

## Use Preprocessed Data to Generate Responses

In [60]:
project_df_labeled = pd.read_csv('data/project_df_labeled.csv')

sampled_df = create_stratified_sample(
    project_df_labeled, 
    n_yes = 10, 
    n_no = 10
)

sampled_df_no_label = sampled_df.drop(columns=['True_Search_Label'])

### Generate responses for original and new filters (Run `filters_eval.ipynb` for new filters)

In [61]:
# Original filters
os.makedirs('predictions', exist_ok=True)
from search_eval_utils import prompt_v1

process_and_save_prompt_output(gemini_client=gemini_client, 
                               model_id=MODEL_ID, 
                               prompt_function=prompt_v1, 
                               df=sampled_df_no_label, 
                               searches_df=pd.read_csv('data/searches.csv'),
                               query_column='FILTER', 
                               output_filename='predictions/v1_stratified.jsonl')

100%|██████████| 172/172 [03:08<00:00,  1.10s/it]


In [62]:
# New filters
from search_eval_utils import prompt_v1
process_and_save_prompt_output(gemini_client=gemini_client, 
                               model_id=MODEL_ID, 
                               prompt_function=prompt_v1, 
                               df=sampled_df_no_label, 
                               searches_df=pd.read_csv('data/searches.csv'),
                               query_column='NEW_FILTER', 
                               output_filename='predictions/v1_stratified_new_filter.jsonl')

100%|██████████| 172/172 [03:11<00:00,  1.11s/it]


In [45]:
from sklearn.preprocessing import LabelBinarizer 

# Read results
responses_df = sampled_df.copy()  
version_responses = ['v1_stratified', 'v1_stratified_new_filter']

# Load df versions 
for version in version_responses:
    version_df = jsonl_to_df(f'predictions/{version}.jsonl')
    version_df.rename({'Project_related_to_Search': f'Project_related_to_Search_{version}', 
                       'Reasoning': f'Reasoning_{version}'}, axis=1, inplace=True)
    responses_df = responses_df.merge(version_df, on=['ProjectID','SEARCH'], how='left')

related_cols = responses_df.filter(regex='related|True_Search_Label').columns
responses_df.dropna(subset=related_cols, inplace=True)

# Encode the relevance columns 
lb = LabelBinarizer()
lb.fit(["NO", "YES"])
for col in related_cols: 
    responses_df[f"{col}_label"] = lb.transform(responses_df[col])

In [48]:
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn import metrics
import numpy as np
from tabulate import tabulate

related_cols_labels = [f"{col}_label" for col in related_cols] # add _label suffix
search_cols = responses_df['SEARCH'].unique()

# Print metrics for each version
for col in related_cols_labels: 
    if col != 'True_Search_Label_label': 

        print(col) 
        # Init metrics table
        metrics_data = [] 
        headers = ["Search", "F1", "Precision", "Recall"]

        # Calculate and add overall metrics
        f1_overall = f1_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan)
        precision_overall = precision_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan)
        recall_overall = recall_score(responses_df['True_Search_Label_label'], responses_df[col], zero_division=np.nan) 
        metrics_data.append(["Overall", f1_overall, precision_overall, recall_overall])

        # Calculate and add metrics for each search
        for search in search_cols:

            # Filter the DataFrame for the current search
            search_df = responses_df[responses_df['SEARCH'] == search]
            
            # Calculate metrics for the current search
            f1 = f1_score(search_df['True_Search_Label_label'], search_df[col], zero_division=np.nan)
            precision = precision_score(search_df['True_Search_Label_label'], search_df[col], zero_division=np.nan)
            recall = recall_score(search_df['True_Search_Label_label'], search_df[col], zero_division=np.nan)
            metrics_data.append([search, f1, precision, recall])
        
        # Print the metrics table
        table = tabulate(metrics_data, headers=headers, tablefmt="grid")
        print(table, "\n")

Project_related_to_Search_v1_stratified_label
+----------------+----------+-------------+----------+
| Search         |       F1 |   Precision |   Recall |
+================+==========+=============+==========+
| Overall        | 0.538462 |        0.49 | 0.597561 |
+----------------+----------+-------------+----------+
| Ceilings       | 0.583333 |        0.5  | 0.7      |
+----------------+----------+-------------+----------+
| Core           | 0.666667 |        0.5  | 1        |
+----------------+----------+-------------+----------+
| Cultured Stone | 0.266667 |        0.4  | 0.2      |
+----------------+----------+-------------+----------+
| Drywall        | 0.642857 |        0.5  | 0.9      |
+----------------+----------+-------------+----------+
| EIFS           | 0        |        0    | 0        |
+----------------+----------+-------------+----------+
| FRP            | 0        |        0    | 0        |
+----------------+----------+-------------+----------+
| Fry Reglet     | 

In [103]:
v1 = responses_df[['ProjectID', 'SEARCH','True_Search_Label','Project_related_to_Search_v1_stratified','Reasoning_v1_stratified']]
print(v1[v1["SEARCH"] == 'Ceilings'].to_markdown())

|    |   ProjectID | SEARCH   | True_Search_Label   | Project_related_to_Search_v1_stratified   | Reasoning_v1_stratified                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
|---:|------------:|:---------|:--------------------|:------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------